In [21]:
import requests
import re
import json

url = "https://examine.com/supplements/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

response = requests.get(url, headers=headers)

# Examine uses Next.js, so the page is rendered dynamically by JavaScript.
# However, the raw data is embedded in the HTML inside the <script> tags as strings!
# We can use Regex to parse these embedded JSON objects directly from the raw HTML text:

# The raw text contains objects like: {\"name\":\"Vitamin D\",\"slug\":\"vitamin-d\",\"url\":\"/supplements/vitamin-d/\"}
matches = re.finditer(r'\\"name\\":\\"(.*?)\\",\\"slug\\":\\"(.*?)\\",\\"url\\":\\"(.*?(?=/supplements/)?/supplements/.*?/)\\"', response.text)

links_list = []
seen_urls = set()

for m in matches:
    name = m.group(1).encode('utf-8').decode('unicode_escape') # Handle any unicode characters like \u0026
    slug = m.group(2)
    relative_url = m.group(3)
    
    # We only want actual supplement links
    if relative_url.startswith('/supplements/') and relative_url != '/supplements/':
        full_url = f"https://examine.com{relative_url}"
        
        if full_url not in seen_urls:
            links_list.append((name, full_url))
            seen_urls.add(full_url)

total_hits = len(set([item[0] for item in links_list]))
print(f"Total hits: {total_hits}")
print(f"Total links: {len(links_list)}")

# Print first five just to verify
# for item in links_list[:5]:
#     print(f"{item[0]}: {item[1]}")

Total hits: 542
Total links: 542


In [ ]:
import sqlite3

# Connect to the SQLite database (this creates the file if it doesn't exist)
db_path = 'examine_catalogue.db'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Create table if it doesn't exist
cursor.execute('''
    CREATE TABLE IF NOT EXISTS supplements (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        url TEXT UNIQUE NOT NULL
    )
''')

# Insert the links into the database
# Using INSERT OR IGNORE to prevent duplicates if the script is run multiple times
cursor.executemany('''
    INSERT OR IGNORE INTO supplements (name, url)
    VALUES (?, ?)
''', links_list)

# Commit the changes and close the connection
conn.commit()

# Verify how many records are in the database total
cursor.execute('SELECT COUNT(*) FROM supplements')
total_records = cursor.fetchone()[0]

conn.close()

print(f"Successfully saved to {db_path}!")
print(f"Total supplements in database: {total_records}")

In [ ]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import re
import time

# Connect to database
db_path = "examine_catalogue.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Get all supplements
cursor.execute('SELECT id, name, url FROM supplements')
all_supplements = cursor.fetchall()

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

# We look for amounts, an optional separator (-, –, 'to'), an optional second amount, and units
dosage_pattern = r'\b(\d+(?:,\d+)?(?:\.\d+)?)\s*(?:-|–|to)?\s*(\d+(?:,\d+)?(?:\.\d+)?)?\s*(mg|g|mcg|iu)\b'

print(f"Starting dosage extraction for {len(all_supplements)} supplements...")

count = 0
for supp_id, name, url in all_supplements:
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(resp.text, 'html.parser')
        
        dosage_text = None
        min_dose = None
        max_dose = None
        min_unit = None
        max_unit = None
        
        dosage_section = soup.find(id=re.compile("dosage-information", re.IGNORECASE))
        
        if dosage_section:
            # Remove interactive buttons and disclaimer popups to clean up text
            for btn in dosage_section.find_all('button'):
                btn.decompose()
                
            # Extract the clean text 
            dosage_text = dosage_section.get_text(separator=' ', strip=True)
            # remove the title itself if it was included in the section
            dosage_text = re.sub(r'(?i)^dosage information\s*', '', dosage_text).strip()
            
            # Search text for min and max numbers
            matches = re.findall(dosage_pattern, dosage_text.lower())
            
            if matches:
                units_dict = {}
                for match in matches:
                    val1 = float(match[0].replace(',', ''))
                    unit = match[2]
                    
                    if unit not in units_dict:
                        units_dict[unit] = set()
                    units_dict[unit].add(val1)
                    
                    if match[1]:
                        val2 = float(match[1].replace(',', ''))
                        units_dict[unit].add(val2)
                
                # We select the first matched unit for simplicity 
                # (most supplements only use one measurement scale)
                best_unit = list(units_dict.keys())[0]
                values = units_dict[best_unit]
                
                if values:
                    min_dose = min(values)
                    max_dose = max(values)
                    min_unit = best_unit
                    max_unit = best_unit

        # Update the database
        cursor.execute('''
            UPDATE supplements 
            SET dosage = ?, min_dose = ?, min_dose_unit = ?, max_dose = ?, max_dose_unit = ?
            WHERE id = ?
        ''', (dosage_text, min_dose, min_unit, max_dose, max_unit, supp_id))
        
        count += 1
        
        # Print progress and commit every 25 rows
        if count % 25 == 0:
            print(f"Processed {count} / {len(all_supplements)}...")
            conn.commit()
            
        # Quick sleep to avoid hammering Examine's servers
        time.sleep(0.4)
        
    except Exception as e:
        print(f"Error processing {name}: {e}")

# Final commit and close
conn.commit()
conn.close()
print("Finished updating dosage information for all supplements!")

In [ ]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import time

# Extract the overview text from all supplements and save to DB
db_path = "examine_catalogue.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Get all supplements
cursor.execute('SELECT id, name, url FROM supplements')
all_supplements = cursor.fetchall()

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

print(f"Starting overview extraction for {len(all_supplements)} supplements...")

count = 0
for supp_id, name, url in all_supplements:
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(resp.text, 'html.parser')
        
        overview_text = None
        
        # Locate the div with the "overview" class
        overview_div = soup.find('div', class_=lambda c: c and 'overview' in c.split())
        
        if overview_div:
            # Extract the text
            overview_text = overview_div.get_text(separator=' ', strip=True)

        # Update the database
        cursor.execute('''
            UPDATE supplements 
            SET overview = ?
            WHERE id = ?
        ''', (overview_text, supp_id))
        
        count += 1
        
        # Print progress and commit every 25 rows
        if count % 25 == 0:
            print(f"Processed {count} / {len(all_supplements)}...")
            conn.commit()
            
        # Quick sleep to avoid hammering Examine's servers
        time.sleep(0.4)
        
    except Exception as e:
        print(f"Error processing {name}: {e}")

# Final commit and close
conn.commit()
conn.close()
print("Finished updating overview information for all supplements!")

In [20]:
import time
import requests
import sqlite3
import re
from bs4 import BeautifulSoup

# Helper function to clean the JSON HTML payload that nextjs uses
def clean_json_html(text):
    text = text.replace('\\u003c', '<').replace('\\u003e', '>').replace('\\"', '"').replace('\\n', ' ')
    soup = BeautifulSoup(text, 'html.parser')
    return soup.get_text(separator=' ', strip=True)

conn = sqlite3.connect('examine_catalogue.db')
c = conn.cursor()

# Process all supplements where the new FAQ sections have not yet been extracted
c.execute("SELECT id, url FROM supplements WHERE what_it_is IS NULL OR benefits IS NULL")
rows = c.fetchall()

total = len(rows)
print(f"Beginning FAQ extraction for {total} catalogs.")

for index, (row_id, url) in enumerate(rows, 1):
    time.sleep(0.4)
    # print(f"Processing ({index}/{total}): {url}")
    
    try:
        resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}, timeout=10)
        
        faq_data = {}
        target_questions = ['What is', 'what is', 'What are', 'what are', 'How does', 'how does']
        
        for line in resp.text.splitlines():
            if not any(q in line for q in target_questions):
                continue
                
            matches = re.finditer(r'\\"(What is.*?|What are.*?|How does.*?)\\",.*?\\"quick_summary\\":\\"(.*?)(?<!\\\\)\\"', line)
            
            for m in matches:
                title = m.group(1).replace('\\u0027', "'").lower()
                summary_raw = m.group(2)
                clean_text = clean_json_html(summary_raw)
                
                # Assign to correct columns based on the question prefix
                if 'what is' in title or 'what are' in title:
                    if 'benefit' in title:
                        faq_data['benefits'] = clean_text
                    elif 'drawback' in title:
                        faq_data['drawbacks'] = clean_text
                    else:
                        faq_data['what_it_is'] = clean_text
                elif 'how does' in title:
                    faq_data['how_it_works'] = clean_text
        
        # update the database if we actually matched items
        if faq_data:
            c.execute('''
                UPDATE supplements
                SET what_it_is = ?, benefits = ?, drawbacks = ?, how_it_works = ?
                WHERE id = ?
            ''', (
                faq_data.get('what_it_is'),
                faq_data.get('benefits'),
                faq_data.get('drawbacks'),
                faq_data.get('how_it_works'),
                row_id
            ))
            
    except requests.exceptions.RequestException as e:
        print(f"Request failed for {url}: {e}")
        
    if index % 20 == 0:
        print(f"Commit point ({index}/{total})...")
        conn.commit()

# Final commit and finish
conn.commit()
print("FAQ Parsing Complete.")
conn.close()


Beginning FAQ extraction for 505 catalogs.
Commit point (20/505)...
Commit point (40/505)...
Commit point (60/505)...
Commit point (80/505)...
Commit point (100/505)...
Commit point (120/505)...
Commit point (140/505)...
Commit point (160/505)...
Commit point (180/505)...
Commit point (200/505)...
Commit point (220/505)...
Commit point (240/505)...
Commit point (260/505)...
Commit point (280/505)...
Commit point (300/505)...
Commit point (320/505)...
Commit point (340/505)...
Commit point (360/505)...
Commit point (380/505)...
Commit point (400/505)...
Commit point (420/505)...
Commit point (440/505)...
Commit point (460/505)...
Commit point (480/505)...
Commit point (500/505)...
FAQ Parsing Complete.


In [22]:
import sqlite3
import requests
import re
import time
from bs4 import BeautifulSoup

# Helper function to clean the JSON HTML payload that nextjs uses
def clean_json_html(text):
    if not text: return ""
    text = text.replace('\\u003c', '<').replace('\\u003e', '>').replace('\\"', '"').replace('\\n', ' ')
    soup = BeautifulSoup(text, 'html.parser')
    return soup.get_text(separator=' ', strip=True)

# Helper function to resolve chunk reference IDs from Next.js payloads
def resolve_dollar_ref(ref_id, full_text):
    if not ref_id or not ref_id.startswith('$'): 
        return ref_id
    
    ref_id = ref_id.strip('$')
    
    # Prefix could be quotes or html closing tags inside the JSON string
    pattern = r'(?:\"|\\"|\\u003e|\[)' + ref_id + r':T[0-9a-fA-F]+,'
    match1 = re.search(pattern, full_text)
    if match1:
        idx = match1.end()
        # Look for the next generic text push
        next_push_idx = full_text.find('__next_f.push([1,"', idx)
        if next_push_idx != -1:
            start_str = next_push_idx + 18
            end_str = full_text.find('"])</script>', start_str)
            if end_str != -1:
                return full_text[start_str:end_str]
                
    # fallback to just direct json if NextJS chunks changes format
    pattern2 = r'(?:\"|\\")' + ref_id + r'(?:\"|\\"):\s*(?:\"|\\")(.*?)(?:\"|\\")'
    match2 = re.search(pattern2, full_text)
    if match2:
        return match2.group(1)

    return "" # Could not resolve

conn = sqlite3.connect('examine_catalogue.db')
c = conn.cursor()

# Process all supplements with missing info or dollar-sign parsed info
query = """
    SELECT id, url 
    FROM supplements 
    WHERE what_it_is IS NULL 
       OR what_it_is = '' 
       OR what_it_is LIKE '$%' 
       OR benefits LIKE '$%'
"""
c.execute(query)
rows = c.fetchall()

total = len(rows)
print(f"Beginning secondary data fix & summary fallback for {total} catalogs.")

for index, (row_id, url) in enumerate(rows, 1):
    time.sleep(0.4)
    
    try:
        resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}, timeout=10)
        
        faq_data = {}
        target_questions = ['What is', 'what is', 'What are', 'what are', 'How does', 'how does']
        found_questions_in_stream = False
        
        # Stream parser
        for line in resp.text.splitlines():
            if not any(q in line for q in target_questions):
                continue
                
            matches = re.finditer(r'\\"(What is.*?|What are.*?|How does.*?)\\",.*?\\"quick_summary\\":\\"(.*?)(?<!\\\\)\\"', line)
            
            for m in matches:
                found_questions_in_stream = True
                title = m.group(1).replace('\\u0027', "'").lower()
                summary_raw = m.group(2)
                
                # Resolve dollar references to actual text chunks
                if summary_raw.startswith('$'):
                    summary_raw = resolve_dollar_ref(summary_raw, resp.text)
                
                clean_text = clean_json_html(summary_raw)
                
                # Assign to correct columns based on the question prefix
                if 'what is' in title or 'what are' in title:
                    if 'benefit' in title:
                        faq_data['benefits'] = clean_text
                    elif 'drawback' in title:
                        faq_data['drawbacks'] = clean_text
                    else:
                        faq_data['what_it_is'] = clean_text
                elif 'how does' in title:
                    faq_data['how_it_works'] = clean_text

        # If we failed to find any FAQ text in the dynamically rendered stream (e.g. no expandable overviews) 
        if not found_questions_in_stream or ('what_it_is' not in faq_data and 'benefits' not in faq_data):
            soup = BeautifulSoup(resp.text, 'html.parser')
            # Look for the fallback summary div
            summary_div = soup.find(id='summary')
            if summary_div:
                summary_text = summary_div.get_text(separator=' ', strip=True)
                # Cleaning up any leading 'Summary ' text that might have been copied from a header 
                summary_text = re.sub(r'(?i)^summary\s*', '', summary_text).strip()
                if summary_text:
                    faq_data['what_it_is'] = summary_text
        
        # Apply fixes securely
        if faq_data:
            c.execute('''
                UPDATE supplements
                SET what_it_is = COALESCE(?, what_it_is), 
                    benefits = COALESCE(?, benefits), 
                    drawbacks = COALESCE(?, drawbacks), 
                    how_it_works = COALESCE(?, how_it_works)
                WHERE id = ?
            ''', (
                faq_data.get('what_it_is'),
                faq_data.get('benefits'),
                faq_data.get('drawbacks'),
                faq_data.get('how_it_works'),
                row_id
            ))
            
    except requests.exceptions.RequestException as e:
        print(f"Request failed for {url}: {e}")
        
    if index % 20 == 0:
        print(f"Commit point ({index}/{total})...")
        conn.commit()

# Final commit and finish
conn.commit()
print("Secondary Database Fix Complete.")
conn.close()


Beginning secondary data fix & summary fallback for 385 catalogs.
Commit point (20/385)...
Commit point (40/385)...
Commit point (60/385)...
Commit point (80/385)...
Commit point (100/385)...
Commit point (120/385)...
Commit point (140/385)...
Commit point (160/385)...
Commit point (180/385)...
Commit point (200/385)...
Commit point (220/385)...
Commit point (240/385)...
Commit point (260/385)...
Commit point (280/385)...
Commit point (300/385)...
Commit point (320/385)...
Commit point (340/385)...
Commit point (360/385)...
Commit point (380/385)...
Secondary Database Fix Complete.
